# The Smoothing Trap: Can We Reconstruct the Sunspot Cycle Without a Model?

## Part I: Reconstructing the 11-Year Sunspot Cycle

In [3]:
import numpy as np
from matplotlib import pyplot as plt

In [4]:
# read data from the .txt file
sunspot_year, sunspot_month, sunspot = np.loadtxt("SN_m_tot_V2.0.txt", unpack=True)

# establish a decimal year time axis
sunspot_time = sunspot_year + (sunspot_month - 1) / 12

In [5]:
# apply a 13-month running mean, taking the comment into account
def smooth_by_running_mean(data, time):
    n = len(time)
    running_mean_13 = data.copy()

    for i in range(0, 6):
        running_mean_13[i] = np.mean(data[0 : i + 7])

    for i in range(6, n - 6):
        mean = (data[i-6] + data[i+6]) / 24

        for j in range(i-5, i+6):
            mean += data[j] / 12

        running_mean_13[i] = mean

    for i in range(n - 6, n):
        running_mean_13[i] = np.mean(data[i - 6 : n])

    return running_mean_13

sunspot_number_running_mean_smoothing = smooth_by_running_mean(sunspot, sunspot_time)

In [6]:
# develop the 13-month optimal running mean for the sunspot record
BETA = 0.01

def smooth_by_optimal_running_mean():
    return 

In [7]:
N = 13
m = N // 2

# build the 13x13 five-diagonal matrix A 
A = np.zeros((N, N))
for row in range(N):
    for offset, val in [(-2, 1), (-1, -4), (0, 6), (1, -4), (2, 1)]:
        col = row + offset 
        if 0 <= col < N:
            A[row, col] = val

# edge corrections: the first and the last rows are [1,-2,1], the second and the second-to-last are [-2,5,-4,1]
A[0, 0] = A[-1, -1] = 1
A[1, 1] = A[-2, -2] = 5
A[0, 1] = A[1, 0] = -2
A[-1, -2] = A[-2, -1] = -2

# adding BETA to the main diagonal
A += BETA * np.eye(N)

# finding the inverse matrix
A_inv = np.linalg.inv(A)

# getting the middle (7th) row
C = A_inv[m, :]          

n = len(sunspot)
smoothed = np.empty(n)
for i in range(n):
    lower_lim = i - m
    upper_lim = i + m
    if lower_lim < 0 or upper_lim > n - 1:
        smoothed[i] = sunspot[max(lower_lim, 0):min(upper_lim, n - 1) + 1].mean()
    else:
        smoothed[i] = BETA * C @ sunspot[lower_lim:upper_lim + 1]

np.savetxt("sunspot_optimal_13m_noD.txt",
           np.column_stack([sunspot_year, sunspot_month, sunspot_time, sunspot, smoothed]),
           fmt=["%d", "%d", "%.4f", "%.1f", "%.2f"],
           header="year month decimal_year raw smoothed_13m_optimal")